# NIDS nhẹ trên UNSW-NB15 — bản chạy lại sau phản biện (v2)

Notebook này sinh **toàn bộ số liệu, bảng và hình của bài báo từ một lần chạy duy nhất**, có cố định seed và bật chế độ tính toán tất định.

So với bản cũ, bản này:
1. Khớp bộ mã hóa đặc trưng phân loại **chỉ trên tập huấn luyện**; giá trị chưa gặp ở tập kiểm thử được gán mã "khác" (góp ý 2.1).
2. Dùng đúng ngưỡng chọn trên validation (bản cũ gõ cứng 0,45).
3. Lấy ma trận nhầm lẫn validation thật cho phép đối chứng tái trọng số (góp ý 2.3).
4. Tính độ nhạy của ngưỡng hiệu chỉnh khi tỉ lệ lớp ước lượng sai (góp ý 2.4).
5. Tạo **4 cấu hình nén cho từng seed trong 10 seed** và báo cáo ΔF1 ghép cặp (góp ý 2.6).
6. So sánh các cấu hình **ở cùng mức báo động giả** (góp ý mục 3).

**Cách chạy:** Runtime → Change runtime type → **CPU** → Runtime → **Run all**. Mất khoảng 30–45 phút.
Chạy xong: **File → Download → Download .ipynb** (để giữ output) và tải tệp `ket_qua_v2.zip` ở ô cuối.

## 0. Cài đặt, cấu hình, môi trường

In [ ]:
!pip install -q kagglehub[pandas-datasets] pyarrow

In [ ]:
import os, sys, json, time, gzip, shutil, platform, warnings, datetime, glob, zipfile
import numpy as np, pandas as pd
import tensorflow as tf
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             roc_auc_score, average_precision_score, balanced_accuracy_score, roc_curve)
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

# ---------------- Cấu hình thực nghiệm ----------------
SEED_MAIN   = 42
SEEDS       = [42, 123, 2024, 7, 99, 256, 512, 1001, 3407, 8888]
EPOCHS      = 50          # mô hình cơ sở, có early stopping (patience 5)
PRUNE_EPOCHS = 15         # tinh chỉnh sau cắt tỉa
SPARSITY    = 0.5
BATCH       = 256
VAL_SIZE    = 0.15
N_REP       = 300         # số mẫu đại diện cho lượng tử hóa INT8
N_BOOT      = 2000
N_TIME_DEFAULT = 200      # đo thời gian với bộ thông dịch mặc định
N_TIME, N_WARM = 500, 50  # đo một luồng / bốn luồng
THR_GRID_TEST = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
THR_GRID_VAL  = [round(x, 2) for x in np.arange(0.05, 0.951, 0.05)]
FPR_TARGETS   = [0.05, 0.10]
RUN_OLD_ENCODER_CHECK = True   # so sánh với cách mã hóa cũ (khớp trên train ∪ test)

OUT = 'ket_qua_v2'
os.makedirs(OUT, exist_ok=True)
tf.config.experimental.enable_op_determinism()

def set_seed(s):
    tf.keras.utils.set_random_seed(s)   # đặt seed cho Python, NumPy và TensorFlow

R = {}   # toàn bộ kết quả dùng cho bài báo
R['moi_truong'] = {
    'ngay_chay': datetime.datetime.now().isoformat(timespec='seconds'),
    'python': sys.version.split()[0], 'tensorflow': tf.__version__, 'numpy': np.__version__,
    'pandas': pd.__version__, 'scikit_learn': sklearn.__version__, 'nen_tang': platform.platform(),
    'gpu': [d.name for d in tf.config.list_physical_devices('GPU')],
    'so_nhan_cpu': os.cpu_count(), 'seeds': SEEDS,
}
print(json.dumps(R['moi_truong'], indent=2, ensure_ascii=False))
if R['moi_truong']['gpu']:
    print('\n*** CẢNH BÁO: đang dùng GPU. Bài báo mô tả chạy trên CPU — hãy đổi Runtime sang CPU rồi chạy lại. ***')

In [ ]:
!pip freeze > ket_qua_v2/requirements.txt
!lscpu | head -20 > ket_qua_v2/cpu_info.txt
print(open('ket_qua_v2/cpu_info.txt').read())

## 1. Tải dữ liệu UNSW-NB15 (dhoogla/unswnb15 trên Kaggle)

In [ ]:
import kagglehub
kagglehub.login()   # bỏ qua nếu đã có kaggle.json hoặc biến môi trường KAGGLE_USERNAME / KAGGLE_KEY

In [ ]:
dataset_path = kagglehub.dataset_download('dhoogla/unswnb15')
files_all = glob.glob(os.path.join(dataset_path, '**', '*.parquet'), recursive=True)
f_train = [f for f in files_all if 'train' in os.path.basename(f).lower()][0]
f_test  = [f for f in files_all if 'test'  in os.path.basename(f).lower()][0]
train_df = pd.read_parquet(f_train)
test_df  = pd.read_parquet(f_test)

In [ ]:
LABEL = 'label'
cot_goc = list(train_df.columns)
co_id = 'id' in cot_goc
DROP = [c for c in ['id', 'attack_cat'] if c in cot_goc]
train_df = train_df.drop(columns=DROP)
test_df  = test_df.drop(columns=[c for c in DROP if c in test_df.columns])
FEATS = [c for c in train_df.columns if c != LABEL]
CAT = [c for c in FEATS if train_df[c].dtype == 'object' or str(train_df[c].dtype) == 'category']

R['du_lieu'] = {
    'kaggle_dataset': 'dhoogla/unswnb15',
    'phien_ban': os.path.basename(os.path.normpath(dataset_path)) if 'dataset_path' in globals() else None,
    'tep_huan_luyen': os.path.basename(f_train) if 'f_train' in globals() else None,
    'tep_kiem_thu': os.path.basename(f_test) if 'f_test' in globals() else None,
    'so_cot_goc': len(cot_goc), 'cot_goc': cot_goc, 'co_cot_id': co_id, 'cot_bi_loai': DROP,
    'so_dac_trung': len(FEATS), 'dac_trung': FEATS, 'dac_trung_phan_loai': CAT,
    'n_train': int(len(train_df)), 'n_test': int(len(test_df)),
    'train_attack': int(train_df[LABEL].sum()), 'train_normal': int((train_df[LABEL] == 0).sum()),
    'test_attack': int(test_df[LABEL].sum()), 'test_normal': int((test_df[LABEL] == 0).sum()),
}
print(json.dumps({k: v for k, v in R['du_lieu'].items() if k not in ('cot_goc',)}, indent=1, ensure_ascii=False))

## 2. Tiền xử lý — bộ mã hóa chỉ khớp trên tập huấn luyện

In [ ]:
def ma_hoa(train_df, test_df, cach='train'):
    # cach='train': khớp chỉ trên train, giá trị chưa gặp -> mã 'khác' (= số loại); cach='hop': cách cũ (train ∪ test)
    tr, te, bao_cao = train_df.copy(), test_df.copy(), {}
    for c in CAT:
        a, b = tr[c].astype(str), te[c].astype(str)
        loai = sorted(a.unique()) if cach == 'train' else sorted(pd.concat([a, b]).unique())
        mp = {v: i for i, v in enumerate(loai)}
        khac = len(loai)
        chua_gap = ~b.isin(mp.keys())
        bao_cao[c] = {'so_loai_train': int(a.nunique()), 'so_gia_tri_chua_gap': int(b[chua_gap].nunique()),
                      'so_dong_chua_gap': int(chua_gap.sum()), 'vi_du': sorted(b[chua_gap].unique().tolist())[:20]}
        tr[c] = a.map(mp).astype(int)
        te[c] = b.map(mp).fillna(khac).astype(int)
    return tr, te, bao_cao

def chuan_bi(tr, te):
    Xf = tr[FEATS].values.astype('float32'); yf = tr[LABEL].values.astype(int)
    Xt = te[FEATS].values.astype('float32'); yt = te[LABEL].values.astype(int)
    sc = StandardScaler().fit(Xf)
    Xf, Xt = sc.transform(Xf).astype('float32'), sc.transform(Xt).astype('float32')
    Xtr, Xv, ytr, yv = train_test_split(Xf, yf, test_size=VAL_SIZE, random_state=42, stratify=yf)
    return Xtr, Xv, Xt, ytr, yv, yt

tr_enc, te_enc, bc = ma_hoa(train_df, test_df, 'train')
X_train, X_val, X_test, y_train, y_val, y_test = chuan_bi(tr_enc, te_enc)
NF = X_train.shape[1]
R['tien_xu_ly'] = {'gia_tri_chua_gap_o_test': bc,
                   'kich_thuoc': {'train_con': len(y_train), 'validation': len(y_val), 'test': len(y_test)}}
print(json.dumps(R['tien_xu_ly'], indent=1, ensure_ascii=False))

## 3. Hàm dùng chung: mô hình, cắt tỉa, chuyển TFLite, suy luận, chỉ số

In [ ]:
def build_model(nf):
    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(nf,)),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')])
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return m

def train_baseline(seed, Xtr, ytr, Xv, yv, verbose=0):
    set_seed(seed)
    m = build_model(Xtr.shape[1])
    es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    h = m.fit(Xtr, ytr, validation_data=(Xv, yv), epochs=EPOCHS, batch_size=BATCH, callbacks=[es], verbose=verbose)
    return m, h

class GiuMask(tf.keras.callbacks.Callback):
    # giữ các trọng số đã cắt tỉa bằng 0 sau mỗi batch
    def __init__(self, masks): super().__init__(); self.masks = masks
    def on_train_batch_end(self, batch, logs=None):
        for layer, mk in self.masks: layer.kernel.assign(layer.kernel * mk)

def prune(base, seed, Xtr, ytr, Xv, yv, verbose=0):
    set_seed(seed)
    m = build_model(Xtr.shape[1]); m.set_weights(base.get_weights())
    masks = []
    for layer in m.layers:
        if isinstance(layer, tf.keras.layers.Dense):
            w = layer.kernel.numpy()
            mk = (np.abs(w) > np.percentile(np.abs(w), SPARSITY * 100)).astype('float32')
            layer.kernel.assign(w * mk); masks.append((layer, tf.constant(mk)))
    es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    m.fit(Xtr, ytr, validation_data=(Xv, yv), epochs=PRUNE_EPOCHS, batch_size=BATCH,
          callbacks=[GiuMask(masks), es], verbose=verbose)
    for layer, mk in masks: layer.kernel.assign(layer.kernel * mk)   # bảo đảm sau restore_best_weights
    w_all = m.get_weights()
    ty_le_0 = float(sum((w == 0).sum() for w in w_all) / sum(w.size for w in w_all))
    return m, ty_le_0

def to_tflite(model, mode, Xrep=None):
    cv = tf.lite.TFLiteConverter.from_keras_model(model)
    if mode == 'fp16':
        cv.optimizations = [tf.lite.Optimize.DEFAULT]; cv.target_spec.supported_types = [tf.float16]
    elif mode == 'int8':
        def rep():
            for i in range(min(N_REP, len(Xrep))): yield [Xrep[i:i+1].astype(np.float32)]
        cv.optimizations = [tf.lite.Optimize.DEFAULT]; cv.representative_dataset = rep
        cv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        cv.inference_input_type = tf.float32; cv.inference_output_type = tf.float32
    return cv.convert()

def luu(b, path):
    with open(path, 'wb') as f: f.write(b)
    with open(path, 'rb') as fi, gzip.open(path + '.gz', 'wb') as fo: shutil.copyfileobj(fi, fo)
    return os.path.getsize(path), os.path.getsize(path + '.gz')

def du_doan(path, X):
    # suy luận theo lô (kết quả trùng suy luận từng mẫu; kiểm tra ở mục 4)
    it = tf.lite.Interpreter(model_path=path)
    i_d = it.get_input_details()[0]
    it.resize_tensor_input(i_d['index'], [len(X), X.shape[1]]); it.allocate_tensors()
    it.set_tensor(i_d['index'], X.astype(np.float32)); it.invoke()
    return it.get_tensor(it.get_output_details()[0]['index']).ravel().astype(np.float64)

def du_doan_tung_mau(path, X):
    it = tf.lite.Interpreter(model_path=path); it.allocate_tensors()
    i_d, o_d = it.get_input_details()[0], it.get_output_details()[0]
    out = np.empty(len(X))
    for i in range(len(X)):
        it.set_tensor(i_d['index'], X[i:i+1].astype(np.float32)); it.invoke()
        out[i] = it.get_tensor(o_d['index']).ravel()[0]
    return out

def chi_so(y, p, t=0.5):
    yp = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0, 1]).ravel()
    return {'nguong': float(t), 'accuracy': accuracy_score(y, yp), 'precision': precision_score(y, yp, zero_division=0),
            'recall_attack': recall_score(y, yp), 'recall_normal': tn / (tn + fp), 'f1': f1_score(y, yp),
            'balanced_accuracy': balanced_accuracy_score(y, yp), 'bao_dong_gia_pct': 100 * fp / (tn + fp),
            'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)}

def tpr_tai_fpr(y, p, fpr):
    # ngưỡng nhỏ nhất sao cho tỉ lệ báo động giả <= fpr; trả về (ngưỡng, recall attack, FPR thực)
    s0 = np.sort(p[y == 0])
    k = int(np.floor(fpr * len(s0)))
    t = np.nextafter(s0[len(s0) - k - 1], np.inf) if k < len(s0) else s0[0]
    yp = (p >= t)
    return float(t), float(yp[y == 1].mean()), float(yp[y == 0].mean())

def do_tg(path, X, n_luong, n=N_TIME, warm=N_WARM):
    it = tf.lite.Interpreter(model_path=path, num_threads=n_luong); it.allocate_tensors()
    i_d, o_d = it.get_input_details()[0], it.get_output_details()[0]
    M = X[:n + warm].astype(np.float32)
    for i in range(warm): it.set_tensor(i_d['index'], M[i:i+1]); it.invoke()
    t = []
    for i in range(warm, warm + n):
        t0 = time.perf_counter(); it.set_tensor(i_d['index'], M[i:i+1]); it.invoke(); _ = it.get_tensor(o_d['index'])
        t.append((time.perf_counter() - t0) * 1000)
    t = np.array(t)
    return {'tb': t.mean(), 'trung_vi': np.percentile(t, 50), 'p95': np.percentile(t, 95)}

def do_tg_mac_dinh(path, X, n=N_TIME_DEFAULT):
    it = tf.lite.Interpreter(model_path=path); it.allocate_tensors()
    i_d, o_d = it.get_input_details()[0], it.get_output_details()[0]
    t0 = time.perf_counter()
    for i in range(n):
        it.set_tensor(i_d['index'], X[i:i+1].astype(np.float32)); it.invoke(); _ = it.get_tensor(o_d['index'])
    return (time.perf_counter() - t0) / n * 1000

CFG = ['Baseline (fp32)', 'Quantized float16', 'Quantized INT8', 'Pruned (50%)', 'Pruned + Quant fp16']

def tao_5_cau_hinh(base, seed, thu_muc, verbose=0):
    os.makedirs(thu_muc, exist_ok=True)
    pm, ty_le_0 = prune(base, seed, X_train, y_train, X_val, y_val, verbose)
    models = {CFG[0]: to_tflite(base, 'fp32'), CFG[1]: to_tflite(base, 'fp16'),
              CFG[2]: to_tflite(base, 'int8', X_train), CFG[3]: to_tflite(pm, 'fp32'), CFG[4]: to_tflite(pm, 'fp16')}
    paths, sizes = {}, {}
    for k, b in models.items():
        p = os.path.join(thu_muc, k.replace(' ', '_').replace('(', '').replace(')', '').replace('%', 'pct').replace('+', 'plus') + '.tflite')
        sizes[k] = luu(b, p); paths[k] = p
    return paths, sizes, ty_le_0

## 4. Lần chạy chính (seed 42) — Bảng 1, 2, 3, 4 và Hình 2, 3, 4

In [ ]:
t_bat_dau = time.time()
base, hist = train_baseline(SEED_MAIN, X_train, y_train, X_val, y_val, verbose=2)
paths, sizes, ty_le_0 = tao_5_cau_hinh(base, SEED_MAIN, os.path.join(OUT, 'mo_hinh_seed42'), verbose=2)
P = {k: du_doan(p, X_test) for k, p in paths.items()}
p_val = du_doan(paths[CFG[0]], X_val)

# kiểm tra suy luận theo lô trùng suy luận từng mẫu
for k, p in paths.items():
    d = np.abs(du_doan_tung_mau(p, X_test[:500]) - P[k][:500]).max()
    print(f'{k:22s} chênh lệch lô/từng mẫu tối đa = {d:.2e}')
R['chay_chinh'] = {'seed': SEED_MAIN, 'so_epoch_co_so': len(hist.history['loss']),
                   'ty_le_tham_so_bang_0_sau_cat_tia_pct': 100 * ty_le_0, 'so_tham_so': int(base.count_params())}
print(R['chay_chinh'], f'— {time.time()-t_bat_dau:.0f}s')

In [ ]:
# Bảng 1: hiệu năng 5 cấu hình (ngưỡng 0,5) + AUC
b1 = []
for k in CFG:
    m = chi_so(y_test, P[k]); m['auc_roc'] = roc_auc_score(y_test, P[k]); m['cau_hinh'] = k; b1.append(m)
B1 = pd.DataFrame(b1).set_index('cau_hinh')
B1.to_csv(f'{OUT}/bang1_hieu_nang.csv'); R['bang1'] = B1.reset_index().to_dict('records')
display(B1[['accuracy', 'precision', 'recall_attack', 'f1', 'recall_normal', 'auc_roc']].round(4))

In [ ]:
# Bảng 2: bootstrap ghép cặp ΔF1 so với cơ sở
rng = np.random.default_rng(42)
n = len(y_test); idx = rng.integers(0, n, size=(N_BOOT, n))
pred = {k: (P[k] >= 0.5).astype(np.int8) for k in CFG}
def f1n(yt, yp):
    tp = np.sum((yp == 1) & (yt == 1)); fp = np.sum((yp == 1) & (yt == 0)); fn = np.sum((yp == 0) & (yt == 1))
    return 2 * tp / (2 * tp + fp + fn)
bf = {k: np.array([f1n(y_test[ii], pred[k][ii]) for ii in idx]) for k in CFG}
b2 = []
for k in CFG[1:]:
    d = bf[k] - bf[CFG[0]]
    lo, hi = np.percentile(d, [2.5, 97.5])
    p2 = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    b2.append({'cau_hinh': k, 'dF1': f1n(y_test, pred[k]) - f1n(y_test, pred[CFG[0]]), 'ktc_duoi': lo, 'ktc_tren': hi,
               'p': p2, 'so_mau_du_doan_khac_co_so': int((pred[k] != pred[CFG[0]]).sum())})
B2 = pd.DataFrame(b2).set_index('cau_hinh'); B2.to_csv(f'{OUT}/bang2_bootstrap.csv'); R['bang2'] = B2.reset_index().to_dict('records')
display(B2.round(5))

In [ ]:
# Bảng 3: kích thước (byte, MiB = 2^20 byte) và thời gian với bộ thông dịch mặc định
b3 = []
g0 = sizes[CFG[0]][1]
for k in CFG:
    raw, gz = sizes[k]
    b3.append({'cau_hinh': k, 'raw_byte': raw, 'gzip_byte': gz, 'raw_MiB': raw / 2**20, 'gzip_MiB': gz / 2**20,
               'giam_raw_pct': 100 * (1 - raw / sizes[CFG[0]][0]), 'giam_gzip_pct': 100 * (1 - gz / g0),
               'suy_luan_mac_dinh_ms': do_tg_mac_dinh(paths[k], X_test)})
B3 = pd.DataFrame(b3).set_index('cau_hinh'); B3.to_csv(f'{OUT}/bang3_kich_thuoc.csv'); R['bang3'] = B3.reset_index().to_dict('records')
display(B3.round(5))

In [ ]:
# Bảng 4: thời gian một luồng (trung vị, P95) và bốn luồng (trung bình)
b4 = []
for k in CFG:
    a = do_tg(paths[k], X_test, 1); b = do_tg(paths[k], X_test, 4)
    b4.append({'cau_hinh': k, 'mot_luong_trung_vi_ms': a['trung_vi'], 'mot_luong_p95_ms': a['p95'],
               'mot_luong_tb_ms': a['tb'], 'bon_luong_tb_ms': b['tb']})
B4 = pd.DataFrame(b4).set_index('cau_hinh')
B4['so_voi_co_so_pct'] = 100 * (B4['mot_luong_trung_vi_ms'] / B4.loc[CFG[0], 'mot_luong_trung_vi_ms'] - 1)
B4['bon_luong_cham_hon_lan'] = B4['bon_luong_tb_ms'] / B4['mot_luong_tb_ms']
B4.to_csv(f'{OUT}/bang4_thoi_gian.csv'); R['bang4'] = B4.reset_index().to_dict('records')
display(B4.round(5))

## 5. Bảng 5–9: điểm vận hành của mô hình cơ sở

In [ ]:
pT = P[CFG[0]]
# Bảng 5: ma trận nhầm lẫn test (0,5); thêm ma trận nhầm lẫn validation (dùng cho Bảng 8)
R['bang5'] = {'test': chi_so(y_test, pT), 'validation': chi_so(y_val, p_val)}
print('Test       TN FP / FN TP:', R['bang5']['test']['TN'], R['bang5']['test']['FP'], '/', R['bang5']['test']['FN'], R['bang5']['test']['TP'])
print('Validation TN FP / FN TP:', R['bang5']['validation']['TN'], R['bang5']['validation']['FP'], '/', R['bang5']['validation']['FN'], R['bang5']['validation']['TP'])

# Bảng 6: quét ngưỡng trên test (chỉ mô tả)
B6 = pd.DataFrame([chi_so(y_test, pT, t) for t in THR_GRID_TEST]).set_index('nguong')
B6.to_csv(f'{OUT}/bang6_quet_nguong_test.csv'); R['bang6'] = B6.reset_index().to_dict('records')
display(B6[['accuracy', 'precision', 'recall_attack', 'recall_normal', 'f1', 'bao_dong_gia_pct']].round(4))

In [ ]:
# Bảng 7: validation vs test
def hang7(y, p, ten):
    m = chi_so(y, p)
    return {'tap': ten, 'ty_le_tan_cong': y.mean(), 'auc_roc': roc_auc_score(y, p), 'auc_pr': average_precision_score(y, p),
            'balanced_accuracy': m['balanced_accuracy'], 'f1': m['f1'], 'accuracy': m['accuracy'],
            'recall_attack': m['recall_attack'], 'recall_normal': m['recall_normal']}
B7 = pd.DataFrame([hang7(y_val, p_val, 'Validation'), hang7(y_test, pT, 'Test')]).set_index('tap')
B7.loc['Chenh_lech'] = B7.loc['Validation'] - B7.loc['Test']
B7.to_csv(f'{OUT}/bang7_val_test.csv'); R['bang7'] = B7.reset_index().to_dict('records')
display(B7.round(4))

# Bảng 8: đối chứng tái trọng số — giữ recall từng lớp của validation, thay tỉ lệ lớp bằng của test
ra, rn = B7.loc['Validation', 'recall_attack'], B7.loc['Validation', 'recall_normal']
pi_tr, pi_te = float(np.concatenate([y_train, y_val]).mean()), float(y_test.mean())
acc_cf = pi_te * ra + (1 - pi_te) * rn
prec_cf = pi_te * ra / (pi_te * ra + (1 - pi_te) * (1 - rn))
f1_cf = 2 * prec_cf * ra / (prec_cf + ra)
f1_v, f1_t = B7.loc['Validation', 'f1'], B7.loc['Test', 'f1']
acc_v, acc_t = B7.loc['Validation', 'accuracy'], B7.loc['Test', 'accuracy']
R['bang8'] = {'recall_attack_val': ra, 'recall_normal_val': rn, 'bao_dong_gia_val_pct': 100 * (1 - rn),
              'acc_du_kien': acc_cf, 'f1_du_kien': f1_cf, 'precision_du_kien': prec_cf,
              'giam_f1_tong': f1_v - f1_t, 'giam_f1_do_ty_le_lop': f1_v - f1_cf,
              'ty_trong_ty_le_lop_trong_giam_f1_pct': 100 * (f1_v - f1_cf) / (f1_v - f1_t),
              'giam_acc_tong': acc_v - acc_t, 'giam_acc_do_ty_le_lop': acc_v - acc_cf,
              'pi_train': pi_tr, 'pi_test': pi_te}
print(json.dumps(R['bang8'], indent=1))

In [ ]:
# Bảng 9: các cách chọn ngưỡng + độ nhạy theo sai số ước lượng tỉ lệ lớp
f1_val_grid = {t: f1_score(y_val, (p_val >= t).astype(int)) for t in THR_GRID_VAL}
best_t = max(f1_val_grid, key=f1_val_grid.get)          # ngưỡng thực sự chọn trên validation
def t_sao(pi, pi_tr=pi_tr):
    r1, r0 = pi / pi_tr, (1 - pi) / (1 - pi_tr); return r0 / (r1 + r0)
t_star = t_sao(pi_te)
t_best_test = float(B6['f1'].idxmax())
B9 = pd.DataFrame([dict(chi_so(y_test, pT, 0.5), cach='Mặc định'),
                   dict(chi_so(y_test, pT, best_t), cach='Tối ưu F1 trên validation'),
                   dict(chi_so(y_test, pT, t_star), cach='Hiệu chỉnh theo tỉ lệ lớp (dùng π_test)'),
                   dict(chi_so(y_test, pT, t_best_test), cach='Tốt nhất trên lưới tại test (tham chiếu)')]).set_index('cach')
B9.to_csv(f'{OUT}/bang9_cach_chon_nguong.csv')
R['bang9'] = {'bang': B9.reset_index().to_dict('records'), 'nguong_validation': best_t, 't_star': t_star,
              'r1': pi_te / pi_tr, 'r0': (1 - pi_te) / (1 - pi_tr), 'nguong_tot_nhat_test': t_best_test,
              'f1_val_theo_nguong': f1_val_grid}
display(B9[['nguong', 'accuracy', 'recall_attack', 'recall_normal', 'f1', 'bao_dong_gia_pct']].round(4))

do_nhay = []
for pi_hat in [0.45, 0.50, round(pi_te, 3), 0.60, 0.65, round(pi_tr, 3)]:
    t = t_sao(pi_hat); m = chi_so(y_test, pT, t)
    do_nhay.append({'pi_uoc_luong': pi_hat, 't_star': t, 'f1': m['f1'], 'bao_dong_gia_pct': m['bao_dong_gia_pct'], 'recall_attack': m['recall_attack']})
DN = pd.DataFrame(do_nhay); DN.to_csv(f'{OUT}/do_nhay_t_star.csv'); R['do_nhay_t_star'] = DN.to_dict('records')
display(DN.round(4))

## 5b. Ước lượng tỉ lệ lớp tại nơi triển khai không dùng nhãn tập kiểm thử (Phản biện 2, mục II)

(a) Không dùng nhãn: đếm-phân loại (CC), đếm có hiệu chỉnh (ACC/BBSE, Forman 2008; Lipton và nnk. 2018), trung bình xác suất (PCC), EM (Saerens và nnk. 2002). (b) Gán nhãn một mẫu kiểm tra nhỏ (n lưu lượng ngẫu nhiên) để ước lượng π, áp dụng t* lên phần còn lại; lặp 1.000 lần.

In [ ]:
def danh_gia_nguong(t, y=None, p=None):
    y = y_test if y is None else y; p = pT if p is None else p
    m = chi_so(y, p, t); return {'nguong': t, 'f1': m['f1'], 'bao_dong_gia_pct': m['bao_dong_gia_pct'], 'recall_attack': m['recall_attack'], 'accuracy': m['accuracy']}
tpr_v = ((p_val >= .5) & (y_val == 1)).sum() / (y_val == 1).sum(); fpr_v = ((p_val >= .5) & (y_val == 0)).sum() / (y_val == 0).sum()
cc = float((pT >= .5).mean()); acc_pi = float(np.clip((cc - fpr_v) / (tpr_v - fpr_v), 0, 1)); pcc = float(pT.mean())
pi_em = pi_tr
for _ in range(1000):
    r1, r0 = pi_em / pi_tr, (1 - pi_em) / (1 - pi_tr); q = pT * r1 / (pT * r1 + (1 - pT) * r0)
    if abs(q.mean() - pi_em) < 1e-8: break
    pi_em = float(q.mean())
UL = []
for ten, pi_hat in [('Đếm phân loại (CC)', cc), ('Đếm có hiệu chỉnh (ACC/BBSE)', acc_pi), ('Trung bình xác suất (PCC)', pcc), ('EM (Saerens)', pi_em), ('π_test thực tế (tham chiếu)', pi_te)]:
    UL.append(dict(phuong_phap=ten, pi_uoc_luong=pi_hat, **danh_gia_nguong(t_sao(pi_hat))))
UL = pd.DataFrame(UL); UL.to_csv(f'{OUT}/bang_uoc_luong_pi.csv', index=False)
rng = np.random.default_rng(0); MK = []
for n in [100, 200, 500, 1000]:
    f1s, fprs, pis = [], [], []
    for _ in range(1000):
        idx = rng.choice(len(y_test), n, replace=False); m = np.ones(len(y_test), bool); m[idx] = False
        pih = y_test[idx].mean(); d = danh_gia_nguong(t_sao(pih), y_test[m], pT[m])
        f1s.append(d['f1']); fprs.append(d['bao_dong_gia_pct']); pis.append(pih)
    MK.append({'n': n, 'pi_tb': np.mean(pis), 'pi_dlc': np.std(pis), 'f1_tb': np.mean(f1s), 'f1_p5': np.percentile(f1s, 5), 'f1_p95': np.percentile(f1s, 95),
               'fpr_tb': np.mean(fprs), 'fpr_p5': np.percentile(fprs, 5), 'fpr_p95': np.percentile(fprs, 95)})
MK = pd.DataFrame(MK); MK.to_csv(f'{OUT}/mau_kiem_tra_pi.csv', index=False)
R['uoc_luong_pi'] = {'tpr_val': tpr_v, 'fpr_val': fpr_v, 'khong_nhan': UL.to_dict('records'), 'mau_kiem_tra': MK.to_dict('records')}
display(UL.round(4)); display(MK.round(4))

## 6. So sánh các cấu hình ở cùng mức báo động giả (góp ý mục 3)

In [ ]:
cf = []
for k in CFG:
    row = {'cau_hinh': k, 'auc_roc': roc_auc_score(y_test, P[k])}
    for f in FPR_TARGETS:
        t, tpr, fpr_thuc = tpr_tai_fpr(y_test, P[k], f)
        row[f'recall_attack_tai_FPR{int(f*100)}'] = tpr; row[f'nguong_FPR{int(f*100)}'] = t; row[f'FPR_thuc_{int(f*100)}'] = fpr_thuc
    cf.append(row)
B10 = pd.DataFrame(cf).set_index('cau_hinh'); B10.to_csv(f'{OUT}/bang10_cung_bao_dong_gia.csv'); R['bang10'] = B10.reset_index().to_dict('records')
display(B10.round(4))

## 7. Hình cho bài báo

In [ ]:
plt.rcParams['font.family'] = 'DejaVu Sans'
def luu_hinh(ten):
    plt.tight_layout(); plt.savefig(f'{OUT}/{ten}', dpi=200, bbox_inches='tight', facecolor='white'); plt.show()

# Hình 2: đường cong huấn luyện
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist.history['loss'], label='Train loss'); ax[0].plot(hist.history['val_loss'], label='Validation loss')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss'); ax[0].set_title('(a) Loss qua các epoch'); ax[0].legend()
ax[1].plot(hist.history['accuracy'], label='Train accuracy'); ax[1].plot(hist.history['val_accuracy'], label='Validation accuracy')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Accuracy'); ax[1].set_title('(b) Accuracy qua các epoch'); ax[1].legend(loc='lower right')
luu_hinh('hinh2_duong_cong_huan_luyen.png')

# Hình 3: kích thước trước và sau gzip
fig, ax = plt.subplots(figsize=(9, 4.2)); x = np.arange(len(CFG)); w = 0.36
ax.bar(x - w/2, B3['raw_MiB'], w, label='Chưa nén (raw)'); ax.bar(x + w/2, B3['gzip_MiB'], w, label='Sau nén gzip')
ax.set_xticks(x); ax.set_xticklabels(CFG, fontsize=8, rotation=12, ha='right'); ax.set_ylabel('MiB')
ax.set_title('Kích thước mô hình: trước và sau khi nén gzip'); ax.legend()
luu_hinh('hinh3_kich_thuoc_gzip.png')

# Hình 4: F1, kích thước gzip, thời gian một luồng
fig, ax = plt.subplots(1, 3, figsize=(13, 4.2)); lab = ['Baseline\n(fp32)', 'Quant\nfloat16', 'Quant\nINT8', 'Pruned\n(50%)', 'Pruned+\nQuant fp16']
for a, vals, tt, yl in [(ax[0], B1['f1'], '(a) F1-score (ngưỡng 0,5)', 'F1-score'),
                        (ax[1], B3['gzip_MiB'], '(b) Kích thước sau gzip', 'MiB'),
                        (ax[2], B4['mot_luong_trung_vi_ms'], '(c) Suy luận một luồng (trung vị)', 'ms/mẫu')]:
    a.bar(lab, vals); a.set_title(tt, fontsize=10); a.set_ylabel(yl); a.tick_params(axis='x', labelsize=7.5)
    for i, v in enumerate(vals): a.text(i, v, f'{v:.4f}', ha='center', va='bottom', fontsize=7.5)
ax[0].set_ylim(0, 1)
luu_hinh('hinh4_so_sanh_5_cau_hinh.png')

# Hình 5: đánh đổi theo ngưỡng
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(B6.index, B6['recall_normal'], 'o-', label='Recall (normal)'); ax.plot(B6.index, B6['recall_attack'], 's-', label='Recall (attack)')
ax.plot(B6.index, B6['f1'], '^-', label='F1-score (attack)')
for t, nhan in [(best_t, 'validation'), (t_star, 't*')]:
    ax.axvline(t, ls='--', lw=1, color='gray'); ax.text(t + 0.005, 0.52, f'{nhan}\n{t:.3f}', fontsize=8, color='gray')
ax.set_xlabel('Ngưỡng phân loại'); ax.set_ylabel('Giá trị'); ax.set_ylim(0.5, 1.02); ax.legend(loc='lower left')
luu_hinh('hinh5_danh_doi_nguong.png')

# Hình 6: ROC validation vs test
fig, ax = plt.subplots(figsize=(6, 5))
for y, p, ten in [(y_val, p_val, 'Validation'), (y_test, pT, 'Test')]:
    f_, t_, _ = roc_curve(y, p); ax.plot(f_, t_, lw=2, label=f'{ten} (AUC = {roc_auc_score(y, p):.4f})')
ax.plot([0, 1], [0, 1], '--', color='gray', lw=1); ax.set_xlabel('Tỉ lệ dương tính giả (FPR)'); ax.set_ylabel('Tỉ lệ dương tính thật (TPR)'); ax.legend(loc='lower right')
luu_hinh('hinh6_roc_val_test.png')

## 8. Mười seed: mô hình cơ sở và ΔF1 ghép cặp của 4 cấu hình nén (góp ý 2.6)

Mỗi seed: huấn luyện mô hình cơ sở → tạo 4 cấu hình nén **từ chính mô hình đó** → đánh giá trên test. Mất khoảng 25–35 phút.

In [ ]:
rows = []
t0 = time.time()
for s in SEEDS:
    b, _ = train_baseline(s, X_train, y_train, X_val, y_val)
    pth, sz, _ = tao_5_cau_hinh(b, s, os.path.join(OUT, 'tam', f'seed{s}'))
    pr = {k: du_doan(p, X_test) for k, p in pth.items()}
    f1b = f1_score(y_test, (pr[CFG[0]] >= 0.5).astype(int))
    for k in CFG:
        m = chi_so(y_test, pr[k]); _, tpr10, _ = tpr_tai_fpr(y_test, pr[k], 0.10)
        rows.append({'seed': s, 'cau_hinh': k, 'f1': m['f1'], 'accuracy': m['accuracy'], 'recall_attack': m['recall_attack'],
                     'bao_dong_gia_pct': m['bao_dong_gia_pct'], 'auc_roc': roc_auc_score(y_test, pr[k]),
                     'recall_attack_tai_FPR10': tpr10, 'gzip_byte': sz[k][1], 'dF1_so_voi_co_so': m['f1'] - f1b})
    print(f'seed {s:5d}: F1 cơ sở = {f1b:.4f}  ({time.time()-t0:.0f}s)')
    shutil.rmtree(os.path.join(OUT, 'tam', f'seed{s}'), ignore_errors=True)
S = pd.DataFrame(rows); S.to_csv(f'{OUT}/muoi_seed_chi_tiet.csv', index=False)

def tom_tat(g):
    d = g['dF1_so_voi_co_so']
    return pd.Series({'f1_tb': g['f1'].mean(), 'f1_dlc': g['f1'].std(ddof=1), 'f1_min': g['f1'].min(), 'f1_max': g['f1'].max(),
                      'dF1_tb': d.mean(), 'dF1_dlc': d.std(ddof=1), 'so_seed_dF1_am': int((d < 0).sum()), 'so_seed_dF1_duong': int((d > 0).sum()),
                      'auc_tb': g['auc_roc'].mean(), 'recall_FPR10_tb': g['recall_attack_tai_FPR10'].mean(),
                      'recall_FPR10_dlc': g['recall_attack_tai_FPR10'].std(ddof=1), 'gzip_byte_tb': g['gzip_byte'].mean()})
S10 = S.groupby('cau_hinh').apply(tom_tat).loc[CFG]
S10.to_csv(f'{OUT}/muoi_seed_tom_tat.csv'); R['muoi_seed'] = {'chi_tiet': S.to_dict('records'), 'tom_tat': S10.reset_index().to_dict('records')}
display(S10.round(4))
f1_42 = S[(S.seed == SEED_MAIN) & (S.cau_hinh == CFG[0])]['f1'].iloc[0]
print('Kiểm tra tái lập: F1 seed 42 lần chạy chính =', round(B1.loc[CFG[0], 'f1'], 6), '| trong vòng 10 seed =', round(f1_42, 6))
R['kiem_tra_tai_lap'] = {'f1_chay_chinh': B1.loc[CFG[0], 'f1'], 'f1_seed42_trong_vong_lap': f1_42}

## 9. Đối chiếu với cách mã hóa cũ (khớp trên train ∪ test)

In [ ]:
if RUN_OLD_ENCODER_CHECK:
    tr_o, te_o, _ = ma_hoa(train_df, test_df, 'hop')
    Xa, Xb, Xc, ya, yb, yc = chuan_bi(tr_o, te_o)
    b_old, _ = train_baseline(SEED_MAIN, Xa, ya, Xb, yb)
    p_old = b_old.predict(Xc, batch_size=8192, verbose=0).ravel()
    p_new = base.predict(X_test, batch_size=8192, verbose=0).ravel()
    R['ma_hoa_cu_vs_moi'] = {'f1_cu': f1_score(yc, (p_old >= .5).astype(int)), 'f1_moi': f1_score(y_test, (p_new >= .5).astype(int)),
                             'auc_cu': roc_auc_score(yc, p_old), 'auc_moi': roc_auc_score(y_test, p_new)}
    print(json.dumps(R['ma_hoa_cu_vs_moi'], indent=1))

## 10. Lưu toàn bộ kết quả và đóng gói

In [ ]:
def to_py(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
with open(f'{OUT}/ket_qua_tong_hop.json', 'w', encoding='utf-8') as f:
    json.dump(R, f, ensure_ascii=False, indent=1, default=to_py)
shutil.rmtree(os.path.join(OUT, 'tam'), ignore_errors=True)
with zipfile.ZipFile('ket_qua_v2.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(glob.glob(f'{OUT}/**/*', recursive=True)):
        if os.path.isfile(p): z.write(p)
print('Đã tạo ket_qua_v2.zip — tải về: thanh Files bên trái → chuột phải ket_qua_v2.zip → Download')
print('Nhớ tải cả notebook: File → Download → Download .ipynb')

In [ ]:
try:
    from google.colab import files
    files.download('ket_qua_v2.zip')
except Exception as e:
    print('Tải thủ công từ thanh Files nếu trình duyệt chặn:', e)